Домашнее задание: градиентный бустинг

Для желающих:

Реализуйте класс GBMClassifier, чтобы ваш бустинг мог решать не только задачу регрессии. Если сделаете рабочий классификатор, предыдущие пункты можно не выполнять.

В качестве доказательства правильности своей реализации сравните результаты обучения вашего бустинга с результатами библиотечного алгоритма на каком-либо датасете. Результаты не обязаны в точности совпадать, допустимым является отклонение вниз на 0.05 по метрике R^2.

Датасет - load_breast_cancer.

In [13]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

Реализация класса GBMClassifier:

In [14]:
class MyGBMClassifier:
    def __init__(self, n_estimators = 100, max_depth = 3, learning_rate = 0.1, subsample = 1, colsample_bytree = 1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree


    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)

        p0 = y.mean()
        self.init_pred = np.log(p0 / (1 - p0)) #логарифм соотношения классов как базовая оценка (позже улучшается с помощью бустинга)

        self.trees = [] # список деревьев и случайного набора фич, на которых обычается конкретно это дерево

        n_samples = len(X)     # количество объектов
        n_features = X.shape[1]    # количество характеристик объекта

        self.feature_importances_ = np.zeros(n_features)
        F = np.full(len(y), self.init_pred)

        for i in range(self.n_estimators):

            p = self._sigmoid(F)  # предсказание (от -1 до 1)
            residuals = y - p # разница между предсказанием и правильным ответом

            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            idx = np.random.choice(n_samples, size=(n_samples * self.subsample), replace=False) # обучаем дерево не на всем датасете, а на случайном подпространстве объектов (строки)
            col_idx = np.random.choice(n_features, size=(n_features * self.colsample_bytree), replace=False) # случайно выбираем столбцы для обучения (фичи)
            X_sub = X[np.ix_(idx, col_idx)]
            residuals_sub = residuals[idx]

            tree.fit(X_sub, residuals_sub) # обучаем дерево предсказывать разницу между правильным ответом и предыдушим предсказанием (residuals_sub - только по выбранным столбцам/характеристикам)
            self.trees.append((tree, col_idx))
            self.feature_importances_[col_idx] += tree.feature_importances_ * self.learning_rate
            F += self.learning_rate * tree.predict(X[:, col_idx])  #  обновляем предсказание

        self.feature_importances_ /= self.feature_importances_.sum() # нормировка чтоб сумма была 1


    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)


    def predict_proba(self, X):
        X = np.array(X)
        F = np.full(len(X), self.init_pred)

        for tree, col_idx in self.trees:
            F += self.learning_rate * tree.predict(X[:, col_idx])
        return self._sigmoid(F)


    def _sigmoid(self, x): # биективное отображение всей числовай прямой в диапазон от 0 до 1 (вероятность)
        return 1 / (1 + np.exp(-x))

    def _encode(self, X, fit = False): # обработка категориальных признаков с помощью One Hot Encoder
        X = pd.DataFrame(X).copy()

        cat_cols = [col for col in X.columns if X[col].dtype == 'object' or X[col].dtype == 'category']

        if fit==True:

            self.ohe_ =  {}
            for col in cat_cols:
                dummies = pd.get_dummies(X[col], prefix=col)
                self.ohe_[col] = dummies.columns
                X = pd.concat([X.drop(col, axis=1), dummies], axis=1)

        else:

            for col in cat_cols:
                dummies = pd.get_dummies(X[col], prefix=col)
                dummies = dummies.reindex(columns=self.ohe_[col], fill_value=0) # на случай если некоторые категориальные значения не встречались в train (в теcтовой выборке они могут встретиться)

                X = pd.concat([X.drop(col, axis=1), dummies], axis=1)

        return np.array(X, dtype=float)




In [15]:
from sklearn.metrics import accuracy_score, roc_auc_score

# моя модель
my_model = MyGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.1)
my_model.fit(X_train, y_train)
my_preds = my_model.predict(X_test)
my_probs = my_model.predict_proba(X_test)

# XGBoost с библиотечной реализацией
xgb_model = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, verbosity=0)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]


results = pd.DataFrame({
    'Модель':   ['MyGBMClassifier', 'XGBoost'],
    'Accuracy': [accuracy_score(y_test, my_preds),   accuracy_score(y_test, xgb_preds)],
    'ROC-AUC':  [roc_auc_score(y_test, my_probs),    roc_auc_score(y_test, xgb_probs)],
})
print(results.round(4))

            Модель  Accuracy  ROC-AUC
0  MyGBMClassifier    0.9561   0.9853
1          XGBoost    0.9561   0.9931


In [16]:
print(my_preds)
print(my_probs)

print(my_model.feature_importances_)

[1 0 0 1 1 0 0 0 0 1 1 0 1 0 1 0 1 1 1 0 1 1 0 1 1 1 1 1 1 0 1 1 1 1 1 1 0
 1 0 1 1 0 1 1 1 1 1 1 1 1 0 0 1 1 1 1 1 0 0 1 1 0 0 1 1 1 0 0 1 1 0 0 1 0
 1 1 1 1 1 1 0 1 1 0 0 0 0 0 1 1 1 1 1 1 1 1 0 0 1 0 0 1 0 0 1 1 1 0 0 1 0
 1 1 0]
[0.90696602 0.11027968 0.11027968 0.90696602 0.90696602 0.11027968
 0.11027968 0.26164462 0.39370659 0.90696602 0.87204227 0.11027968
 0.90907118 0.24762768 0.90696602 0.11027968 0.90907118 0.90696602
 0.90696602 0.11027968 0.89584166 0.90696602 0.11027968 0.90696602
 0.87204227 0.9090671  0.90696602 0.87204227 0.90696602 0.11027968
 0.88692046 0.90696602 0.77225086 0.90907118 0.90696602 0.90696602
 0.27371694 0.90696602 0.11027968 0.86325962 0.90696602 0.11027968
 0.90696602 0.90696602 0.90015787 0.90696602 0.73640876 0.87204227
 0.90696602 0.90696602 0.11027968 0.11027968 0.87865981 0.9090671
 0.90696602 0.90696602 0.90696602 0.11027968 0.12675955 0.90696602
 0.90696602 0.11027968 0.11027968 0.87204227 0.90696602 0.87204227
 0.11027968 0.11027968 0.906966